# Guidance: validate the new boundary with a linear stand-in (issue #2, step 1)

Goal: reuse your own `a_ufl` / `L_ufl` linear formulation almost unchanged, and add a
**second, independent linear** loss coefficient on the boundary you intend to use for
radiation later. This validates *which faces get selected* before any non-linearity or
Newton solver is involved at all.

Do **not** implement `T**4` here -- that happens in `src/surroptim/weak_form/` and in
`notebooks/main_nonlinear.ipynb`, once this stand-in has confirmed the face selection.


In [1]:
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import mesh, fem
import dolfinx.fem.petsc  # NOTE: must be imported explicitly (see issue history:
                          # `from dolfinx import fem` alone does not expose fem.petsc).


## Minimal problem (flat rectangle, no mechanics) to isolate the boundary term

Same `(r, z)` rectangle convention as your code, trimmed down: no mechanics, no
sensors, no I/O. Just enough to see the effect of the new boundary term on `uh`.


In [2]:
width, thickness = 1.0, 0.5
nx, ny = 60, 20

domain = mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([0.0, 0.0]), np.array([width, thickness])],
    [nx, ny],
    cell_type=mesh.CellType.triangle,
)
V = fem.functionspace(domain, ("CG", 1))


## NEW: tag boundaries instead of relying on untagged `ufl.ds`

Your current code integrates `advection_coeff * du * v * r * ufl.ds` over the **whole**
boundary and relies on the Dirichlet lifting to silently override the `z=0`
contribution afterwards. That is fine as long as every boundary term is the *same*
linear expression everywhere.

It stops being fine the moment you want a *different* term (radiative vs none) on
different faces -- which is exactly issue #2's open "Gamma_D status" question. Tagging
makes that choice an explicit, visible one-liner instead of an implicit side effect of
how Dirichlet BCs are applied.


In [3]:
def boundary_bottom(x):
    return np.isclose(x[1], 0.0)

def boundary_rest(x):
    return ~np.isclose(x[1], 0.0)

facet_dim = domain.topology.dim - 1
bottom_facets = mesh.locate_entities_boundary(domain, facet_dim, boundary_bottom)
rest_facets = mesh.locate_entities_boundary(domain, facet_dim, boundary_rest)

GAMMA_D, GAMMA_N = 1, 2
facet_indices = np.concatenate([bottom_facets, rest_facets])
facet_markers = np.concatenate([
    np.full_like(bottom_facets, GAMMA_D),
    np.full_like(rest_facets, GAMMA_N),
])
order = np.argsort(facet_indices)
facet_tags = mesh.meshtags(domain, facet_dim, facet_indices[order], facet_markers[order])

ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
# ds(GAMMA_D) integrates ONLY over z=0, ds(GAMMA_N) over everything else.
# `ufl.ds` (no argument) still integrates over the whole boundary, unchanged.

dofs_bottom = fem.locate_dofs_geometrical(V, boundary_bottom)
bcs = [fem.dirichletbc(PETSc.ScalarType(0.0), dofs_bottom, V)]

r_weight = fem.Function(V)
r_weight.interpolate(lambda x: np.maximum(np.abs(x[0]), 1e-14))

uh = fem.Function(V)
u_n = fem.Function(V)
du = ufl.TrialFunction(V)
v = ufl.TestFunction(V)


## The stand-in: a second linear coefficient, only on `Gamma_N`

Deliberately **not** the radiative term -- just large enough that its effect is
visible, so you can confirm `ds(GAMMA_N)` selects the faces you think it selects,
completely independently of Newton / non-linear convergence questions.


In [4]:
dt = 5.0e-3
thermal_capacity, diffusion_coeff, advection_coeff = 1.0, 1.0, 1.0
radiation_stand_in_coeff = 5.0  # arbitrary, > advection_coeff so the effect is visible

def grad_cyl(w):
    return ufl.as_vector([ufl.Dx(w, 0), ufl.Dx(w, 1)])

a_ufl = (
    thermal_capacity * (1.0 / dt) * du * v * r_weight * ufl.dx
    + diffusion_coeff * ufl.dot(grad_cyl(du), grad_cyl(v)) * r_weight * ufl.dx
    + advection_coeff * du * v * r_weight * ds(GAMMA_N)
    + radiation_stand_in_coeff * du * v * r_weight * ds(GAMMA_N)   # <- the new term
)
L_ufl = thermal_capacity * (1.0 / dt) * u_n * v * r_weight * ufl.dx

a = fem.form(a_ufl)
L = fem.form(L_ufl)
A = dolfinx.fem.petsc.assemble_matrix(a, bcs=bcs)
A.assemble()

ksp = PETSc.KSP().create(domain.comm)
ksp.setOperators(A)
ksp.setType("preonly")
ksp.getPC().setType("lu")

u_n.x.array[:] = 5.0  # start "hot" everywhere so the extra loss has something to remove
for step in range(20):
    b = dolfinx.fem.petsc.assemble_vector(L)
    dolfinx.fem.petsc.apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    dolfinx.fem.petsc.set_bc(b, bcs)
    ksp.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()
    u_n.x.array[:] = uh.x.array

print("mean T after 20 steps, WITH stand-in term:", uh.x.array.mean())


mean T after 20 steps, WITH stand-in term: 0.31039940969139995


## Sanity check to run yourself

Set `radiation_stand_in_coeff = 0.0` above, rerun this notebook, and compare the mean
temperature: with the extra loss term active it should end up **lower** (more heat
removed from `Gamma_N`) than with it set to zero. If it does not move, or moves on the
wrong faces, the tagging above is wrong -- fix that *before* moving to
`notebooks/main_nonlinear.ipynb`, where the same `ds(GAMMA_N)` measure gets reused with
the real radiative term instead of this stand-in.
